# 05 — A demanding case: forest cover type

*Chapter 06 — Random Forests · Notebook 5 of 5*

This is where the chapter comes together. We take everything we built — bagging, the "random" that
decorrelates the trees, out-of-bag scoring, and the estimator's dials — to a large, messy, realistic
problem: predicting which of seven tree species covers a patch of Colorado forest from cartographic
measurements. It is a *random forest* on an actual *forest*, and it is exactly the kind of problem the
method was made for: many features, non-linear structure, and severe class imbalance.

Two questions drive the notebook. **Does the forest win here** — on a genuinely non-linear problem,
where the linear and margin models that beat it on breast cancer (chapter 03 / 05) should struggle?
And can we **evaluate it honestly** when the classes are wildly imbalanced, so accuracy alone would
flatter us?

**Prerequisites.**
- **This chapter, NB 1–4** — the whole forest: bagging, the decorrelation dial `max_features`,
  out-of-bag estimation, and MDI vs permutation importance (which we *named* in NB 4 and put to work
  here).
- **The cross-method spine** — KNN (ch 01), logistic regression (ch 03), SVM (ch 05): on breast cancer
  the linear/margin models won. Here we test the reverse.
- **Module 00** — the confusion matrix, precision / recall, and **macro vs weighted** averaging
  (NB 07); cross-validation (NB 10); the train/test split (NB 04).

**What you'll be able to do.**
- Run an honest forest workflow on a large, imbalanced, non-linear dataset.
- See a forest win decisively where a linear model cannot.
- Evaluate under imbalance with macro-F1, per-class recall, and a confusion matrix — not accuracy alone.
- Read feature importance honestly: MDI vs permutation, and the one-hot dilution trap.
- See why a forest scales gently to large data, and know when to reach for boosting.

In [ ]:
import logging
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import CLASS_CYCLE, COLORS

# Show the one-time dataset fetch (~14 MB on first run, then cached) -- never silenced.
logging.basicConfig(level=logging.INFO)
viz.use_course_style()
SEED = 0

# The seven cover types (Blackard & Dean 1999).
COVER = {
    1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine", 4: "Cottonwood/Willow",
    5: "Aspen", 6: "Douglas-fir", 7: "Krummholz",
}

# Fetch the full dataset (581k x 54), then take a stratified 30k subsample to stay brisk.
X_full, y_full = fetch_covtype(return_X_y=True, as_frame=True)
X, _, y, _ = train_test_split(
    X_full, y_full, train_size=30000, stratify=y_full, random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
feature_names = list(X.columns)
n_cont = sum(not c.startswith(("Wilderness", "Soil")) for c in feature_names)
n_wild = sum(c.startswith("Wilderness") for c in feature_names)
n_soil = sum(c.startswith("Soil") for c in feature_names)
print(f"full covtype: {X_full.shape[0]} patches; subsample {X.shape[0]} "
      f"-> train {X_train.shape[0]} / test {X_test.shape[0]}")
print(f"54 features = {n_cont} continuous + {n_wild} wilderness one-hot + {n_soil} soil one-hot")
print("no scaling for the forest (a tree splits on thresholds -- NB 4)")

## Where we are, and the stakes

The dataset describes 581 012 patches of Colorado forest, each 30 m × 30 m, by 54 cartographic
features: ten continuous measurements (elevation, slope, distances to water, roads and fire points,
hillshade), four one-hot wilderness-area columns, and forty one-hot soil-type columns. The label is
the dominant tree species, one of seven. We work with a 30 000-patch stratified subsample so every
cell runs in seconds.

This is a different *shape* of problem from breast cancer. There, two classes sat almost linearly
separated, and a linear/margin model won. Here the species are carved out by **non-linear
interactions** — a species lives at a certain elevation *and* slope *and* soil — and the classes are
**severely imbalanced**. Both facts will matter, and a random forest is built for exactly this.

In [ ]:
fig = viz.plot_class_balance(y_train.map(COVER))
fig.axes[0].set_title("forest cover types in the training set")
fig.axes[0].tick_params(axis="x", rotation=30)
fig.tight_layout()

**Read the figure.** Two species dominate — **Lodgepole Pine (≈49 %)** and **Spruce/Fir
(≈37 %)** — while **Aspen (≈1.6 %)** and **Cottonwood/Willow (≈0.5 %)** are rare. A model that always
predicted "Lodgepole" would already score near 49 % accuracy without learning anything.
Under this imbalance, accuracy is ruled by the two big classes, so we will judge the forest by
**per-class recall** and **macro-F1**, not by accuracy alone.

## The honest workflow, and the cross-method question

We run the workflow module 00 taught: fit on the training set, read the **sealed test once**. We fit
three models — a random forest (no scaling, NB 4), a single decision tree, and a linear model
(logistic regression, which *does* need scaling) — and compare them on the same test set, using fixed
defaults (no tuning on the test set). The forest also reports its **out-of-bag** score as a free
in-training check. The real question is not which knob to turn but which model *family* fits this
problem's shape.

In [ ]:
forest = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
forest.fit(X_train, y_train)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
linear = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
linear.fit(X_train, y_train)

rf_pred = forest.predict(X_test)
acc = {
    "random forest": accuracy_score(y_test, rf_pred),
    "single tree": accuracy_score(y_test, tree.predict(X_test)),
    "logistic reg.": accuracy_score(y_test, linear.predict(X_test)),
}
print({k: round(v, 3) for k, v in acc.items()})
print(f"forest out-of-bag accuracy: {forest.oob_score_:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
bar_colors = [COLORS["highlight"], COLORS["model"], COLORS["model"]]
ax.bar(list(acc), list(acc.values()), color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
for i, v in enumerate(acc.values()):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylabel("test accuracy")
ax.set_ylim(0, 1)
ax.grid(False)
fig.tight_layout()

**Read the figure.** The forest **wins decisively**: about **0.844** against the linear model's
**0.729** — eleven points clear — with the single tree (**0.770**) in between. This is the **reverse
of breast cancer**, where a linear/margin model edged the forest (RF ≈ 0.94 < SVM 0.965 on that
near-linear problem). The species here are separated by non-linear interactions of elevation,
distances and soil; a committee of trees carves those regions, while a single straight boundary
cannot. The forest's **out-of-bag** accuracy (≈ 0.846) lands right on the sealed-test number (≈
0.844), confirming it is not overfit. And note the forest needed **no scaling**, while the linear
model required a `StandardScaler` — the scale-invariance of NB 4, lived. *The right tool tracks the
problem's shape.*

## Accuracy hides imbalance: macro vs weighted

A single accuracy number is dangerous under imbalance. Accuracy — and the **weighted** F1, which
weights each class by how many samples it has — average over *samples*, so the two big classes set the
score and the rare species barely register. The **macro** average instead means over *classes*,
unweighted: a rare class counts exactly as much as a common one. When you care about getting the rare
classes right, report macro.

In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, rf_pred),
    "weighted-F1": f1_score(y_test, rf_pred, average="weighted"),
    "macro-F1": f1_score(y_test, rf_pred, average="macro"),
}
print({k: round(v, 3) for k, v in metrics.items()})

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(list(metrics), list(metrics.values()),
       color=[COLORS["model"], COLORS["model"], COLORS["highlight"]],
       edgecolor=COLORS["text"], linewidth=0.6)
for i, v in enumerate(metrics.values()):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.grid(False)
fig.tight_layout()

**Read the figure.** Accuracy (**0.844**) and weighted-F1 (**0.840**) sit together, both ruled
by the two big classes. Macro-F1 drops to **0.737**. That ten-point gap is the price of the rare
classes the forest is missing — a cost accuracy keeps hidden and macro-F1 brings into the open. The
next figure shows exactly which species pay it.

In [ ]:
labels = list(range(1, 8))
recalls = recall_score(y_test, rf_pred, average=None, labels=labels)
names = [COVER[k] for k in labels]
for name, r in zip(names, recalls, strict=True):
    print(f"  {name:<18} recall {r:.3f}")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bar_colors = [CLASS_CYCLE[i % len(CLASS_CYCLE)] for i in range(len(labels))]
ax.bar(names, recalls, color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
for i, r in enumerate(recalls):
    ax.text(i, r, f"{r:.2f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylabel("recall (RF)")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=30)
ax.grid(False)
fig.tight_layout()

**Read the figure.** The big, common classes are handled well — Lodgepole Pine **0.90**,
Ponderosa **0.87**, Spruce/Fir **0.82** — but the rare ones are hard: **Aspen ≈ 0.28**,
Cottonwood/Willow ≈ 0.56, Douglas-fir ≈ 0.57. The forest catches barely a quarter of the Aspen
patches. A random forest does **not** fix imbalance — it inherits it, because each tree's majority
vote leans toward the classes it saw most. The levers for imbalance are `class_weight='balanced'` or
resampling (you try one in *Your turn*), not more trees.

In [ ]:
cm = confusion_matrix(y_test, rf_pred, labels=labels)
short = ["Spruce", "Lodgepl", "Pondrsa", "Cottonw", "Aspen", "Douglas", "Krummh"]
fig = viz.plot_confusion_matrix(cm, short)
fig.axes[0].set_title("cover type: true (rows) vs predicted (columns)")
fig.tight_layout()

**Read the figure.** The misses are **structured**, not random noise. Read the Aspen row (5):
most true-Aspen patches are predicted **Lodgepole Pine** — the rare class collapses into its common,
look-alike neighbour. Ponderosa and Douglas-fir trade errors with each other. The big classes hold a
strong diagonal. The matrix tells us the forest's weakness here is not carelessness but **class
imbalance and species similarity**: when a rare class sits near a common one, the committee defaults to
the common vote.

## Reading importance honestly: MDI vs permutation

NB 4 *named* permutation importance and deferred it to here. Now we use it. Recall the two readings:
**MDI** (mean decrease in impurity) sums how much each feature reduced impurity across the forest — it
is fast but **biased** toward continuous / high-cardinality features and it **dilutes** a signal
spread across a one-hot group. **Permutation importance** shuffles one column and measures how far the
held-out accuracy drops — a direct test of how much the model *relies* on that feature. When one
feature genuinely dominates, the two readings **agree**; the disagreements are where care is needed.

In [ ]:
mdi = forest.feature_importances_
perm = permutation_importance(
    forest, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1
)

def _group(values, prefix):
    return sum(v for c, v in zip(feature_names, values, strict=True) if c.startswith(prefix))

elev = feature_names.index("Elevation")
print(f"Elevation       MDI {mdi[elev]:.3f}   permutation {perm.importances_mean[elev]:.3f}")
print(f"40 Soil_* total MDI {_group(mdi, 'Soil'):.3f}   "
      f"permutation {_group(perm.importances_mean, 'Soil'):.3f}")

fig, (ax_mdi, ax_perm) = plt.subplots(1, 2, figsize=(14, 5.5))
viz.plot_feature_importances(feature_names, mdi, top=10, ax=ax_mdi, color=COLORS["model"])
ax_mdi.set_title("MDI (impurity) importance")
viz.plot_feature_importances(
    feature_names, perm.importances_mean, std=perm.importances_std, top=10,
    ax=ax_perm, color=COLORS["highlight"],
)
ax_perm.set_title("permutation importance (test)")
fig.tight_layout()

**Read the figure.** **Elevation dominates both** panels — it is the top feature by a wide
margin under MDI and under permutation alike. The two measures are built differently (MDI is a
normalized impurity *share*; permutation is an accuracy *drop*), so their heights are not directly
comparable — but they **agree on the ranking**, because one feature genuinely carries most of the
signal. That is the clean case. NB 4 met the messier one with MDI alone: a single tree's importance
spiked on one of several correlated features (0.74), and the forest *spread* that credit across the
group (peak 0.15) — a reminder that no single bar is "the" important one. Now look for the soil: no
single `Soil_Type_*` column comes anywhere near Elevation (a couple slip into the permutation top-ten
only because the rest of the field is flat). Yet the **forty soil columns combined** score about
**0.14 (MDI) / 0.11 (permutation)** — together the **second-largest** signal in the data, behind only
Elevation. That is the **dilution trap** made vivid: soil type is not unimportant, its signal is split
across forty thin one-hot dummies (and MDI's bias toward many-valued continuous features compounds it).
The lesson: **read importance at the group level**, never one one-hot column at a time.

## A forest at scale

Two practical virtues make a random forest the first model many practitioners reach for on large
tabular data. It gives a generalization estimate **for free** through out-of-bag scoring (no
validation split to carve off, as we saw above: OOB ≈ test). And its training cost grows **gently** with
the number of rows — the opposite of chapter 05's SVM, whose RBF fit-time climbed steeply with `n`.
Let's measure it.

In [ ]:
sizes = [1000, 2000, 4000, 8000, 16000, 32000, 64000]
fit_times = []
for nsz in sizes:
    Xs, _, ys, _ = train_test_split(
        X_full, y_full, train_size=nsz, stratify=y_full, random_state=SEED
    )
    rf_timed = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1)
    t0 = time.perf_counter()
    rf_timed.fit(Xs, ys)
    fit_times.append(time.perf_counter() - t0)

sizes_a = np.array(sizes, dtype=float)
times_a = np.array(fit_times)
exponent = np.polyfit(np.log(sizes_a), np.log(times_a), 1)[0]
ref_linear = times_a[0] * (sizes_a / sizes_a[0]) ** 1.0
ref_svm = times_a[0] * (sizes_a / sizes_a[0]) ** 1.6
print("fit times (s):", [round(t, 2) for t in fit_times])
print(f"empirical exponent ~ n^{exponent:.2f}")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(sizes_a, times_a, marker="o", color=COLORS["model"], linewidth=2,
        label=f"random forest (measured, ~n^{exponent:.2f})")
ax.plot(sizes_a, ref_linear, linestyle="--", color=COLORS["muted"], linewidth=1.5,
        label="n^1.0 reference (linear)")
ax.plot(sizes_a, ref_svm, linestyle=":", color=COLORS["highlight"], linewidth=1.5,
        label="n^1.6 reference (ch 05 SVM RBF)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("training set size n")
ax.set_ylabel("fit time (s, single-thread)")
ax.legend()
fig.tight_layout()

**Read the figure.** The forest's fit-time tracks the **n^1.0** line (measured ≈ n^0.99): double
the data, double the time. The dotted **n^1.6** curve is chapter 05's SVM RBF scaling (worst-case
O(n³)) — by 64 000 rows it towers far above the forest. And because the trees are independent, a forest
is **embarrassingly parallel**: `n_jobs=-1` spreads them across cores for a near-linear speed-up.
Gentle scaling, no required scaling, and a free out-of-bag estimate are why a random forest is the
go-to first model on large tabular data.

## Error analysis, honest limits, and the bridge to boosting

The forest here is **strong**: it wins decisively (0.844), scales roughly linearly, needs no scaling,
and grades itself for free. But the capstone earns its honesty by stating the limits plainly.

- **It won *because* the problem is non-linear.** On near-linear breast cancer the forest *lost* to the
  SVM. No method is best everywhere — the right tool tracks the problem's shape.
- **It inherits class imbalance.** Aspen recall ≈ 0.28; more trees will not fix that. The remedies are
  `class_weight='balanced'`, resampling, or a metric/threshold chosen for the rare class.
- **It is no longer one readable tree.** Chapter 04's payoff — a flowchart a person can audit — is gone;
  hundreds of trees trade interpretability for accuracy and stability.
- **Importance is not causal**, and MDI dilutes across one-hot groups — read it at the group level, and
  cross-check with permutation.

**When to push further:** gradient **boosting** (chapters 07–10) often edges a random forest on tabular
data, by building trees that fix each other's mistakes rather than averaging independent ones — at the
cost of more careful tuning. A forest's claim is not "best", it is **strong with little effort**.

## Your turn

1. **Read the confusion matrix (easy).** From the matrix above, name the class that Aspen is most often
   confused with, and explain — using the class-balance figure — why the forest leans that way.

2. **Fight the imbalance (medium).** Refit the forest with `class_weight='balanced'` and recompute
   macro-F1 and Aspen's recall. Did balancing help the rare classes? What happened to overall accuracy?
   Write one sentence on the trade-off you observe.

3. **Group the importance (harder).** Rank the features by MDI and by permutation importance, then sum
   the forty `Soil_Type_*` columns under each measure. Explain in two sentences why the individual soil
   dummies look unimportant under *both* measures, even though soil type clearly matters.

## What you built

- You ran an **honest forest workflow** on a large, imbalanced, non-linear dataset and saw the forest
  **win decisively** (0.844) where a linear model could not (0.729) — the reverse of breast cancer.
- You evaluated **under imbalance**: accuracy and weighted-F1 (≈ 0.84) hid what **macro-F1** (0.737) and
  **per-class recall** (Aspen ≈ 0.28) revealed, and the **confusion matrix** showed *where* the misses
  go (rare classes collapse into common look-alikes).
- You read **importance honestly**: MDI and permutation **agreed** that Elevation dominates, while the
  forty one-hot soil columns were **diluted** — a lesson to read importance at the group level.
- You measured the forest's **gentle, near-linear scaling**, the counterpoint to chapter 05's SVM wall,
  and saw OOB stand in for a test set, for free.

**Vocabulary you now own:** class imbalance · macro vs weighted F1 · per-class recall · MDI vs
permutation importance · one-hot dilution · embarrassingly parallel · the right-tool-for-the-shape
principle.

## Chapter wrap — random forests, end to end

From a single, wobbly tree to a steady committee, in five notebooks: **averaging cuts variance** (NB 1,
bagging) → **the "random" decorrelates the trees** (NB 2, the ρσ² floor) → **out-of-bag gives a free
estimate** (NB 3) → **the estimator and its dials** (NB 4) → **a demanding case the forest wins**
(NB 5). A random forest is the course's first **ensemble**: many high-variance, low-bias trees averaged
into one strong, low-variance model that needs no scaling, grades itself for free, scales gently, and
runs in parallel — the direct cure for the single tree's variance that closed chapter 04, and the
strong baseline practitioners reach for first on tabular data.

Its limits point forward. A forest averages **independent** trees; it is not always the most accurate,
and it trades away the single tree's readability. The **boosting** family (chapters 07–10) takes the
other path — building trees in sequence, each correcting the last — and often edges the forest on
tabular problems, at the cost of more tuning. That is where we go next.

## Going further (optional)

- **Imbalance head-on:** `class_weight='balanced'` (you try it in *Your turn*), or resampling the rare
  classes, trades a little overall accuracy for much better recall on Aspen and Cottonwood.
- **`ExtraTrees`** (extremely randomized trees) push the "random" one step further — random split
  *thresholds* — often matching a forest a touch faster.
- **OOB-guided selection:** because OOB ≈ test (we saw it here), you can pick `n_estimators` or
  `max_features` from the OOB score alone, without a separate validation split.

## References

- Breiman, L. (2001). *Random Forests.* Machine Learning 45, 5–32.
  DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140.
  DOI: [10.1007/BF00058655](https://doi.org/10.1007/BF00058655)
- Ho, T. K. (1998). *The random subspace method for constructing decision forests.* IEEE TPAMI 20(8),
  832–844. DOI: [10.1109/34.709601](https://doi.org/10.1109/34.709601)
- Strobl, C., Boulesteix, A.-L., Zeileis, A., & Hothorn, T. (2007). *Bias in random forest variable
  importance measures.* BMC Bioinformatics 8, 25.
  DOI: [10.1186/1471-2105-8-25](https://doi.org/10.1186/1471-2105-8-25)
- Blackard, J. A., & Dean, D. J. (1999). *Comparative accuracies of artificial neural networks and
  discriminant analysis in predicting forest cover types from cartographic variables.* Computers and
  Electronics in Agriculture 24(3), 131–151.
  DOI: [10.1016/S0168-1699(99)00046-0](https://doi.org/10.1016/S0168-1699(99)00046-0)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §15 (Random Forests). DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
  (2nd ed.), §8.2. DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 04 — The estimator & its parameters. Next: Module 07 — AdaBoost.*